# Verificación de cifras — Fase A.3

Recalcula desde los datos las cifras que aparecen con valores distintos en la
bitácora archivada (`DECISIONES_ARCHIVO_2026-09.md`), en `DECISIONES.md`, en el
notebook 01 y en el borrador de la monografía. Cada cifra queda en
`reports/00_verificacion_cifras.csv` con el valor verificado, el valor citado
antes y su fuente. **Regla:** la monografía cita solo la columna `valor_verificado`.

Alcance: son cifras de la **Etapa 1** (datos crudos). Las cifras que dependen del
área de estudio se recalculan en `02_preparacion_datos.ipynb` sobre la nueva área (D-018).

In [1]:
import sys
from pathlib import Path


def _encontrar_raiz(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise RuntimeError("No se encontró pyproject.toml")


RAIZ = _encontrar_raiz(Path.cwd())
SRC_DIR = RAIZ / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from trufi_ds.notebook_setup import *  # noqa: F403
from trufi_ds.config import QUERIES_PARQUET, GTFS_DIR

REPORTES = PROJECT_ROOT / "reports"
BBOX_BOLIVIA = {"lat_min": -23.0, "lat_max": -9.0, "lon_min": -70.0, "lon_max": -56.0}

df = pl.read_parquet(QUERIES_PARQUET)
print(f"Filas crudas: {df.height:,}")

Filas crudas: 1,927,675


## 1 · Coordenadas de origen inválidas

In [2]:
lat, lon = pl.col("lat_orig"), pl.col("lon_orig")
es_cero = (lat == 0.0) | (lon == 0.0)
en_bolivia = lat.is_between(BBOX_BOLIVIA["lat_min"], BBOX_BOLIVIA["lat_max"]) & lon.is_between(
    BBOX_BOLIVIA["lon_min"], BBOX_BOLIVIA["lon_max"]
)
n_cero = df.filter(es_cero).height
n_fuera_bolivia = df.filter(~es_cero & ~en_bolivia).height
n_fuera_bolivia_incl_cero = df.filter(~en_bolivia).height
print(f"Origen (0,0): {n_cero}")
print(f"Origen fuera de Bolivia (sin contar ceros): {n_fuera_bolivia}")
print(f"Origen fuera de Bolivia (contando ceros): {n_fuera_bolivia_incl_cero}")

Origen (0,0): 3
Origen fuera de Bolivia (sin contar ceros): 9
Origen fuera de Bolivia (contando ceros): 12


## 2 · Duplicados exactos

`is_duplicated()` marca **todas** las copias de un grupo; al deduplicar se conserva
una por grupo, así que las filas que efectivamente se eliminan son
`filas_marcadas − grupos`.

In [3]:
cols_exacto = [c for c in df.columns if c not in ("source_file", "source_batch", "source_encoding")]
marcadas = df.select(cols_exacto).is_duplicated()
n_marcadas = int(marcadas.sum())
n_grupos = df.filter(marcadas).select(cols_exacto).unique().height
n_eliminar = n_marcadas - n_grupos
print(f"Filas marcadas como duplicado exacto: {n_marcadas}")
print(f"Grupos distintos: {n_grupos}")
print(f"Filas a eliminar (copias sobrantes): {n_eliminar}")

Filas marcadas como duplicado exacto: 104
Grupos distintos: 44
Filas a eliminar (copias sobrantes): 60


## 3 · Celdas H3 r8 con consultas de origen

In [4]:
validas = df.filter(~es_cero & en_bolivia)
celdas_orig = validas.select(
    pl.struct("lat_orig", "lon_orig")
    .map_batches(
        lambda s: pl.Series(
            [h3.latlng_to_cell(a, b, 8) for a, b in zip(s.struct.field("lat_orig"), s.struct.field("lon_orig"))]
        ),
        return_dtype=pl.Utf8,
    )
    .alias("h3_origin")
)["h3_origin"]
n_celdas_orig = celdas_orig.n_unique()
print(f"Celdas H3 r8 con al menos una consulta de origen válida: {n_celdas_orig:,}")

Celdas H3 r8 con al menos una consulta de origen válida: 1,516


## 4 · Usuarios

In [5]:
n_usuarios = validas["userID"].n_unique()
mediana_por_usuario = validas.group_by("userID").len()["len"].median()
print(f"Usuarios únicos (origen válido): {n_usuarios:,}")
print(f"Mediana de consultas por usuario: {mediana_por_usuario}")

Usuarios únicos (origen válido): 130,545
Mediana de consultas por usuario: 5.0


## 5 · GTFS: agencias, líneas, recorridos, paradas

Vocabulario fijado: **línea** = `route` (141); **recorrido** = `trip`/`shape` (626).

In [6]:
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
shapes = pd.read_csv(GTFS_DIR / "shapes.txt", usecols=["shape_id"])
stops = pd.read_csv(GTFS_DIR / "stops.txt")
agency = pd.read_csv(GTFS_DIR / "agency.txt")
gtfs = {
    "agencias": len(agency),
    "lineas_routes": routes["route_id"].nunique(),
    "recorridos_trips": trips["trip_id"].nunique(),
    "recorridos_shapes": shapes["shape_id"].nunique(),
    "paradas": stops["stop_id"].nunique(),
}
print(gtfs)

{'agencias': 43, 'lineas_routes': 141, 'recorridos_trips': 626, 'recorridos_shapes': 626, 'paradas': 22320}


## 6 · Tabla de verificación

In [7]:
filas = [
    ("filas_crudas", df.height, "1.927.675", "borrador §1", ""),
    ("filas_origen_cero", n_cero, "3", "notebook 01 §7", ""),
    ("filas_origen_fuera_bolivia", n_fuera_bolivia, "15", "DECISIONES_ARCHIVO (2026-09-25)",
     "La cifra 15 era de un pipeline anterior; con el bbox Bolivia actual y sin contar (0,0) son las indicadas."),
    ("filas_origen_fuera_bolivia_incl_cero", n_fuera_bolivia_incl_cero, "—", "—", "Incluye las filas (0,0)."),
    ("duplicados_exactos_filas_marcadas", n_marcadas, "104", "notebook 01 §6 / D-013 previo", "Cuenta todas las copias."),
    ("duplicados_exactos_grupos", n_grupos, "—", "—", ""),
    ("duplicados_exactos_filas_eliminadas", n_eliminar, "~52", "docs/00_riesgos", "Cifra que se reporta como eliminada (D-013)."),
    ("celdas_h3r8_con_consultas_origen", n_celdas_orig, "1.523 / 1.516", "DECISIONES_ARCHIVO / notebook 01 §12",
     "1.523 venía del pipeline anterior; la cifra de área de estudio se recalcula en la Etapa 2."),
    ("celdas_h3r8_con_consultas_en_area_gtfs", 950, "950", "notebook 01 §12",
     "Obsoleta: el área GTFS fue superada (D-018). No citar."),
    ("usuarios_unicos_origen_valido", n_usuarios, "130.549", "borrador", ""),
    ("mediana_consultas_por_usuario", mediana_por_usuario, "3", "borrador", ""),
    ("gtfs_agencias", gtfs["agencias"], "43", "notebook 01 §14", ""),
    ("gtfs_lineas_routes", gtfs["lineas_routes"], "141 / 625", "borrador (Introducción usa 625 'rutas')",
     "Unidad fijada: 'línea' = route."),
    ("gtfs_recorridos_trips", gtfs["recorridos_trips"], "626", "notebook 01 §14", "Unidad fijada: 'recorrido' = trip."),
    ("gtfs_recorridos_shapes", gtfs["recorridos_shapes"], "626", "notebook 01 §14", ""),
    ("gtfs_paradas", gtfs["paradas"], "22.320", "notebook 01 §14", ""),
]
tabla = pl.DataFrame(
    [
        {"cifra": c, "valor_verificado": str(v), "valor_citado_previo": p, "fuente_previa": f,
         "coincide": None if p == "—" else str(v).replace(".0", "") in p.replace(".", "").split(" / "),
         "nota": n}
        for c, v, p, f, n in filas
    ]
)
tabla.write_csv(REPORTES / "00_verificacion_cifras.csv")
print(tabla)

shape: (16, 6)
┌─────────────────┬─────────────────┬─────────────────┬────────────────┬──────────┬────────────────┐
│ cifra           ┆ valor_verificad ┆ valor_citado_pr ┆ fuente_previa  ┆ coincide ┆ nota           │
│ ---             ┆ o               ┆ evio            ┆ ---            ┆ ---      ┆ ---            │
│ str             ┆ ---             ┆ ---             ┆ str            ┆ bool     ┆ str            │
│                 ┆ str             ┆ str             ┆                ┆          ┆                │
╞═════════════════╪═════════════════╪═════════════════╪════════════════╪══════════╪════════════════╡
│ filas_crudas    ┆ 1927675         ┆ 1.927.675       ┆ borrador §1    ┆ true     ┆                │
│ filas_origen_ce ┆ 3               ┆ 3               ┆ notebook 01 §7 ┆ true     ┆                │
│ ro              ┆                 ┆                 ┆                ┆          ┆                │
│ filas_origen_fu ┆ 9               ┆ 15              ┆ DECISIONES_ARC ┆ fal